### Imports

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys
import os
import torch
import copy

sys.path.insert(0, os.path.dirname(os.getcwd()))

### Configuration

In [7]:
PROFILE = {
    "DoE_Gesamt_RhodaminB": [25,9],
    "DoE_Gesamt_4MU": [13,19,12]
}
INTERPOLATION_OPTIONS = ["linear", "cubic"]
FILENAMES = ["DoE_Gesamt_4MU", "DoE_Gesamt_RhodaminB"]

### Read Data

In [8]:
from src.bay_op_projekt.helper import load_data_to_df
path = Path(f"../data/{FILENAMES[0]}.csv")
parameter_names = ["Temperatur", "pH"]
target_names = ["Lipase_Aktivität_1", "Lipase_Aktivität_2", "Lipase_Aktivität_3"] # The mean of these rows is used in further calculations.
df, train_X_raw, train_Y_raw = load_data_to_df(path=path, parameter_row_names=parameter_names,target_row_names=target_names)
df = df.drop(PROFILE[FILENAMES[0]]).reset_index(drop=True)

### Normalise Data

In [9]:
from src.bay_op_projekt.normalisation import normalize, standardize_Y

bounds_raw = torch.tensor([
    [8.0, 5.0],   # [min_Temperatur, min_pH]
    [33.0, 8.5]   # [max_Temperatur, max_pH]
])

train_X = normalize(copy.deepcopy(train_X_raw), bounds_raw)
train_Y, Y_mean, Y_std = standardize_Y(train_Y_raw)
train_Yvar = torch.tensor(train_Y_raw / Y_std.item()**2, dtype=torch.double).reshape(-1, 1)
bounds_normalized = torch.zeros(2, 2, dtype=torch.double)
bounds_normalized[1] = 1.0

### Modeltraining

In [ ]:
from src.bay_op_projekt.model_training import train_gp_model

model = train_gp_model(train_X, train_Y, train_Yvar)

GP-Modell trainiert


### Extract the Recomendations from the model.

In [ ]:
from botorch.acquisition import qLogExpectedImprovement
from botorch.optim import optimize_acqf
best_f = train_Y.max()

acq_function = qLogExpectedImprovement(model=model, best_f=best_f)

candidate_normalized, acq_value = optimize_acqf(
    acq_function=acq_function,
    bounds=bounds_normalized,
    q=3,
    num_restarts=30,
    raw_samples=512,
)
candidate_raw = candidate_normalized * (bounds_raw[1] - bounds_raw[0]) + bounds_raw[0]
print(f"\n➡ Nächstes Experiment vorgeschlagen:")
for i in range(3):
    print(f"   Kandidat {i+1}:")
    print(f"   Temperatur: {candidate_raw[i, 0].item():.2f} °C")
    print(f"   pH-Wert:    {candidate_raw[i, 1].item():.3f}")


➡ Nächstes Experiment vorgeschlagen:
   Kandidat 1:
   Temperatur: 33.00 °C
   pH-Wert:    5.000
   Kandidat 2:
   Temperatur: 33.00 °C
   pH-Wert:    8.500
   Kandidat 3:
   Temperatur: 17.82 °C
   pH-Wert:    6.367
